# 🧪 json2quantum
**Convierte un circuito cuántico definido en JSON a un circuito Qiskit.**

### Formato JSON esperado
El campo `controls` acepta **ambos formatos**:
```json
"controls": [0]               // entero directo
"controls": [{"qubit": 0}]   // objeto con clave 'qubit'
```

## 📦 1. Instalación de dependencias

In [ ]:
# Instalar Qiskit si no está disponible (necesario en Colab)
try:
    import qiskit
    print(f"✅ Qiskit {qiskit.__version__} ya instalado")
except ImportError:
    import subprocess, sys
    subprocess.check_call([sys.executable, "-m", "pip", "install",
                           "qiskit", "qiskit-aer", "pylatexenc", "-q"])
    print("✅ Qiskit instalado correctamente")

## 📚 2. Imports

> **Nota**: todos los imports de Qiskit están aquí, a nivel de módulo,
> para que Pylance/el type-checker de VS Code pueda resolver
> los tipos (como `QuantumCircuit`) usados en las firmas de funciones.

In [ ]:
import json
import math
from pathlib import Path

# ── Qiskit core ─────────────────────────────────────────────────────────────
from qiskit import QuantumCircuit, QuantumRegister, ClassicalRegister
from qiskit.circuit.library import (
    RZGate, RYGate, XGate, CXGate, HGate, ZGate, SGate, TGate,
)
from qiskit.visualization import plot_histogram

# ── Simulador ────────────────────────────────────────────────────────────────
try:
    from qiskit_aer import AerSimulator
except ImportError:
    from qiskit.providers.basic_provider import BasicSimulator as AerSimulator  # fallback

import matplotlib.pyplot as plt

print("✅ Imports OK")

## 🔧 3. Parser JSON → Qiskit

### Fixes aplicados
| # | Problema | Solución |
|---|----------|-----------|
| 1 | `Could not find name 'QuantumCircuit'` | Imports movidos al nivel superior (celda 2) |
| 2 | `TypeError: CXGate is immutable` | Se llama `.to_mutable()` antes de asignar `label` |
| 3 | `controls` como enteros (`[0]`) | Parser acepta `int` y `{"qubit": int}` |

### Compuertas soportadas
| Nombre en JSON | Compuerta | Parámetro |
|---|---|---|
| `X`, `H`, `Z`, `S`, `T` | Pauli/Clifford | No |
| `Rz` / `RZ` | RZ(θ) | `parameter` en radianes |
| `Ry` / `RY` | RY(θ) | `parameter` en radianes |
| `CNOT` / `CX` | CX | No (los controles van en `controls`) |

In [ ]:
# ── Mapa: nombre JSON → constructor de compuerta Qiskit ─────────────────────
GATE_MAP: dict = {
    # 1 qubit — sin parámetro
    "X":    lambda p: XGate(),
    "H":    lambda p: HGate(),
    "Z":    lambda p: ZGate(),
    "S":    lambda p: SGate(),
    "T":    lambda p: TGate(),
    # 1 qubit — con parámetro (ángulo en radianes)
    "Rz":   lambda p: RZGate(p),
    "RZ":   lambda p: RZGate(p),
    "Ry":   lambda p: RYGate(p),
    "RY":   lambda p: RYGate(p),
    # Compuertas controladas
    "CNOT": lambda p: CXGate(),
    "CX":   lambda p: CXGate(),
}


def _resolve_control(c) -> int:
    """
    Normaliza un elemento de la lista 'controls'.
    Acepta dos formatos:
      - int directo:        0
      - objeto con clave:   {"qubit": 0}
    """
    if isinstance(c, int):
        return c
    if isinstance(c, dict):
        return c["qubit"]
    raise ValueError(f"Formato de control no reconocido: {c!r}")


def parse_json_to_circuit(json_data: dict) -> QuantumCircuit:
    """
    Convierte un diccionario con el formato json2quantum
    en un QuantumCircuit de Qiskit.

    Parámetros
    ----------
    json_data : dict
        Datos cargados desde el JSON.

    Retorna
    -------
    QuantumCircuit
        Circuito Qiskit listo para simular o visualizar.
    """
    n_qubits   = json_data["qubits"]
    operations = json_data["operations"]
    level      = json_data.get("level", "?")
    basis      = json_data.get("basis", [])

    print(f"📋 Nivel       : {level}")
    print(f"⚛️  Qubits      : {n_qubits}")
    print(f"🔑 Basis        : {basis}")
    print(f"🔢 Operaciones  : {len(operations)}")
    print("─" * 45)

    qr = QuantumRegister(n_qubits, name="q")
    cr = ClassicalRegister(n_qubits, name="c")
    qc = QuantumCircuit(qr, cr, name=f"nivel{level}")

    for i, op in enumerate(operations):
        gate_def  = op["gate"]
        name      = gate_def["name"]
        label     = gate_def.get("label", name)       # opcional: usa name si no hay label
        target    = gate_def["target"]
        parameter = gate_def.get("parameter", None)   # None para compuertas sin ángulo
        controls  = gate_def.get("controls", [])       # acepta int o {"qubit": int}

        # ── validar nombre ──────────────────────────────────────────────────
        if name not in GATE_MAP:
            raise ValueError(
                f"Operación [{i:02d}]: compuerta '{name}' no reconocida.\n"
                f"  Disponibles: {list(GATE_MAP.keys())}"
            )

        # FIX 2: .to_mutable() → crea una copia editable del singleton
        gate = GATE_MAP[name](parameter).to_mutable()
        gate.label = label

        # FIX 3: normalizar controles (int o dict)
        control_indices = [_resolve_control(c) for c in controls]
        all_qubits      = [qr[c] for c in control_indices] + [qr[target]]

        qc.append(gate, all_qubits)

        param_str = f", θ={parameter:.4f} rad" if parameter is not None else ""
        print(f"  [{i:02d}] {name:<6} → q[{target}]{param_str}  controls={control_indices}  ({label})")

    qc.measure(qr, cr)

    print("─" * 45)
    print(f"✅ Profundidad: {qc.depth()} | Operaciones: {qc.count_ops()}")
    return qc


print("✅ Parser definido")

## 📂 4. Cargar el JSON y construir el circuito

In [ ]:
JSON_FILE = "nivel6.json"   # ← cambia aquí para probar otro nivel

with open(JSON_FILE, "r") as f:
    data = json.load(f)

print("📄 JSON cargado:")
print(json.dumps(data, indent=2))

In [ ]:
qc = parse_json_to_circuit(data)

## 🖼️ 5. Visualizar el circuito

In [ ]:
qc_draw = qc.copy()
qc_draw.remove_final_measurements()  # dibujar sin mediciones para mayor claridad

print("🔭 Diagrama del circuito:")
display(qc_draw.draw(output="mpl", style="iqp", fold=-1))

## 🚀 6. Simular con Aer

In [ ]:
SHOTS = 4096

result = AerSimulator().run(qc, shots=SHOTS).result()
counts = result.get_counts()

print(f"🎲 Resultados ({SHOTS} shots):")
for state, count in sorted(counts.items(), key=lambda x: -x[1]):
    bar = "█" * int(count / SHOTS * 40)
    print(f"  |{state}⟩ : {count:5d}  ({count/SHOTS:.1%})  {bar}")

In [ ]:
fig = plot_histogram(
    counts,
    title=f"Distribución de mediciones — nivel {data.get('level', '?')}",
    figsize=(10, 4),
    color="#6929C4",
)
plt.tight_layout()
plt.show()

## 📊 7. Información del circuito

In [ ]:
from qiskit.transpiler.preset_passmanagers import generate_preset_pass_manager

backend = AerSimulator()
pm      = generate_preset_pass_manager(optimization_level=1, backend=backend)
qc_t    = pm.run(qc)

print("📊 Estadísticas del circuito")
print(f"  Profundidad (original)    : {qc.depth()}")
print(f"  Profundidad (transpilado) : {qc_t.depth()}")
print(f"  Número de qubits          : {qc.num_qubits}")
print(f"  Operaciones               : {qc.count_ops()}")
print(f"  Operaciones transpiladas  : {qc_t.count_ops()}")

---
## 🧩 8. Ejemplo inline — Bell State

Prueba rápida sin archivo en disco. Usa el formato nuevo (`controls: [0]`).

In [ ]:
bell_json = {
    "level": 99,
    "qubits": 2,
    "basis": ["H", "CNOT"],
    "operations": [
        {"gate": {"name": "H",    "target": 0, "controls": []}},
        {"gate": {"name": "CNOT", "target": 1, "controls": [0]}},  # int directo ✓
    ]
}

qc_bell = parse_json_to_circuit(bell_json)

qc_bell_draw = qc_bell.copy()
qc_bell_draw.remove_final_measurements()
display(qc_bell_draw.draw(output="mpl", style="iqp"))

counts_bell = AerSimulator().run(qc_bell, shots=2048).result().get_counts()
plot_histogram(counts_bell, title="Bell State |Φ+⟩", color="#00B28A")